# Field model comparison

Evaluate all six field models on the same train/eval split and compare RMSE, MAE, and runtime.

In [4]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in ["oceanbench-core", "oceanbench-models", "oceanbench-bench"]:
    d = ROOT / p
    if d.exists() and str(d) not in sys.path:
        sys.path.insert(0, str(d))

import numpy as np
from oceanbench_core.types import ObservationBatch, QueryPoints
from oceanbench_models.belief.field import (
    LocalLinearFieldModel,
    GPFieldModel,
    SparseOnlineGPFieldModel,
    PseudoInputGPFieldModel,
    GMRFFieldModel,
)
from oceanbench_bench import run_field_model_comparison

In [5]:
rng = np.random.default_rng(42)
n_train, n_eval = 80, 200
def field(lat, lon):
    return 20.0 + 2.0 * np.exp(-0.01 * ((lat - 30)**2 + (lon + 80)**2))

lat_tr = rng.uniform(24, 36, n_train)
lon_tr = rng.uniform(-86, -74, n_train)
obs = ObservationBatch(
    lats=lat_tr, lons=lon_tr,
    values=field(lat_tr, lon_tr) + 0.1 * rng.standard_normal(n_train),
    variable="temp",
)
lat_ev = rng.uniform(24, 36, n_eval)
lon_ev = rng.uniform(-86, -74, n_eval)
qp = QueryPoints(lats=lat_ev, lons=lon_ev)
y_true = field(lat_ev, lon_ev)

In [6]:
models = [
    ("local_linear", LocalLinearFieldModel({"k_neighbors": 15}, seed=42)),
    ("gp", GPFieldModel({"lengthscale": 1.0, "noise": 0.01}, seed=42)),
    ("sparse_online_gp", SparseOnlineGPFieldModel({"max_points": 100}, seed=42)),
    ("pseudo_input_gp", PseudoInputGPFieldModel({"n_pseudo": 50}, seed=42)),
    ("gmrf", GMRFFieldModel({"n_lat": 25, "n_lon": 25}, seed=42)),
]
result = run_field_model_comparison(obs, qp, y_true, models, seed=42)
for run in result.runs:
    print(f"{run.model_name}: RMSE={run.metrics.get('rmse', float('nan')):.4f}, MAE={run.metrics.get('mae', float('nan')):.4f}, fit={run.fit_time_seconds:.3f}s, predict={run.predict_time_seconds:.3f}s")

local_linear: RMSE=0.1216, MAE=0.0858, fit=0.000s, predict=0.001s
gp: RMSE=0.2532, MAE=0.1435, fit=0.002s, predict=0.004s
sparse_online_gp: RMSE=19.5460, MAE=19.5329, fit=0.228s, predict=0.002s
pseudo_input_gp: RMSE=19.6551, MAE=19.6484, fit=0.233s, predict=0.002s
gmrf: RMSE=0.0800, MAE=0.0638, fit=0.003s, predict=0.000s


Discussion: compare which models are faster vs more accurate; note that GP is exact but O(N³), while sparse/pseudo-input and GMRF trade some accuracy for scalability.